### Loading the dataset

The data set was generated using the `simulated.py` script

In [1]:
import pandas as pd
import numpy as np

# Loading dataset into dataframe
df = pd.read_csv('../data/simulated_server_metrics.csv', parse_dates=['timestamp'])

df.head()

,timestamp,server_id,server_type,cpu_percent,memory_percent,disk_io,is_anomaly,anomaly_type
0,2025-01-01 00:00:00,web_1,web,22.483571,23.274928,63.451172,0,normal
1,2025-01-01 00:05:00,web_1,web,19.308678,25.694752,43.981028,0,normal
2,2025-01-01 00:10:00,web_1,web,23.238443,36.885905,57.091144,0,normal
3,2025-01-01 00:15:00,web_1,web,27.615149,35.100183,51.622232,0,normal
4,2025-01-01 00:20:00,web_1,web,18.829233,28.184342,33.647097,0,normal


### Computing the rolling averages
For each row the average of the last N readings are computed for each server. This smooths out noise and shows the recent trends. 

Rolling windows of 5, 10 and 30 are created to capture different times. For example 5 readings = 25 min "very recent", 30 readings = 2.5hrs "much longer".
`min_periods=1` is used for the first 4 rows which would have been `NaN`, this is a tiny fraction of the whole dataset is affected so the issue of less meaningful rows is negliable. 

`.shift` - ensures that the current observation isn't used to calculate the rolling value. For example we have a `cpu_percent` spike **95**, if this was the current observation and included in the rolling value, it could have made the anomaly signal weak and blend into the normal data. This needs to be done for each server type, and ensure that the values used to calculate the rolling mean are not from other server types

In [2]:
# Creating rolling windows for the mean and standard deviation
def add_rolling_features(df,column, windows):
    for window in windows:
        mean_col_name = f'{column}_roll_mean_{window}'
        shifted_values = df.groupby('server_id')[column].shift(1)
        df[mean_col_name] = shifted_values.groupby(df['server_id']).rolling(window=window, min_periods=1).mean().reset_index(level=0, drop=True)
        std_col_name = f'{column}_roll_standard_deviation_{window}'
        df[std_col_name] = shifted_values.groupby(df['server_id']).rolling(window=window, min_periods=1).std().reset_index(level=0, drop=True)
    return df

df = add_rolling_features(df, 'cpu_percent',[5,10,30])
df = add_rolling_features(df, 'memory_percent',[5,10,30])
df = add_rolling_features(df, 'disk_io',[5,10,30])
df.head(30)

,timestamp,server_id,server_type,cpu_percent,memory_percent,disk_io,is_anomaly,anomaly_type,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,...,memory_percent_roll_mean_10,memory_percent_roll_standard_deviation_10,memory_percent_roll_mean_30,memory_percent_roll_standard_deviation_30,disk_io_roll_mean_5,disk_io_roll_standard_deviation_5,disk_io_roll_mean_10,disk_io_roll_standard_deviation_10,disk_io_roll_mean_30,disk_io_roll_standard_deviation_30
0,2025-01-01 00:00:00,web_1,web,22.483571,23.274928,63.451172,0,normal,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-01 00:05:00,web_1,web,19.308678,25.694752,43.981028,0,normal,22.483571,NaN,...,23.274928,NaN,23.274928,NaN,63.451172,NaN,63.451172,NaN,63.451172,NaN
2,2025-01-01 00:10:00,web_1,web,23.238443,36.885905,57.091144,0,normal,20.896125,2.244988,...,24.484840,1.711074,24.484840,1.711074,53.716100,13.767470,53.716100,13.767470,53.716100,13.767470
3,2025-01-01 00:15:00,web_1,web,27.615149,35.100183,51.622232,0,normal,21.676897,2.085378,...,28.618528,7.261269,28.618528,7.261269,54.841115,9.928172,54.841115,9.928172,54.841115,9.928172
4,2025-01-01 00:20:00,web_1,web,18.829233,28.184342,33.647097,0,normal,23.161460,3.422705,...,30.238942,6.756748,30.238942,6.756748,54.036394,8.264545,54.036394,8.264545,54.036394,8.264545
5,2025-01-01 00:25:00,web_1,web,18.829315,34.272010,40.813835,0,normal,22.295015,3.541161,...,29.828022,5.923218,29.828022,5.923218,49.958535,11.591881,49.958535,11.591881,49.958535,11.591881
6,2025-01-01 00:30:00,web_1,web,27.896064,31.706079,48.501220,0,normal,21.564164,3.855648,...,30.568686,5.599921,30.568686,5.599921,45.431067,9.172940,48.434418,11.019753,48.434418,11.019753
7,2025-01-01 00:35:00,web_1,web,23.837174,37.501373,61.122924,0,normal,23.281641,4.464336,...,30.731171,5.130049,30.731171,5.130049,46.335106,9.216942,48.443961,10.059644,48.443961,10.059644
8,2025-01-01 00:40:00,web_1,web,17.652628,30.372548,54.637231,0,normal,23.401387,4.470913,...,31.577446,5.318574,31.577446,5.318574,47.141462,10.482656,50.028832,10.336069,50.028832,10.336069
9,2025-01-01 00:45:00,web_1,web,22.712800,35.128161,38.324992,0,normal,21.408883,4.341611,...,31.443569,4.991256,31.443569,4.991256,47.744461,10.883881,50.540876,9.789777,50.540876,9.789777


In [3]:
# Check the anomaly cpu_percent rolling mean and std
print(
    df[(df['server_id'] == 'web_1') & (df['is_anomaly'] == 1)]
    [['timestamp', 'server_id','cpu_percent', 'is_anomaly', 'cpu_percent_roll_mean_5','cpu_percent_roll_standard_deviation_5']]
    .tail()
)
# Test whether the calculated rolling means and std are within the server types, don't overlap with eachother 
print(
    df[df['server_id'] == 'web_2']
    [['timestamp', 'server_id', 'cpu_percent', 'is_anomaly', 'cpu_percent_roll_mean_5','cpu_percent_roll_standard_deviation_5']]
    .head()
)


               timestamp server_id  cpu_percent  is_anomaly  \
2732 2025-01-10 11:40:00     web_1   100.000000           1   
2733 2025-01-10 11:45:00     web_1   100.000000           1   
2734 2025-01-10 11:50:00     web_1    91.502833           1   
2735 2025-01-10 11:55:00     web_1    96.467007           1   
2736 2025-01-10 12:00:00     web_1    92.788004           1   

      cpu_percent_roll_mean_5  cpu_percent_roll_standard_deviation_5  
2732                90.915210                               6.313477  
2733                94.223492                               5.736910  
2734                95.262219                               6.310350  
2735                96.640544                               3.552186  
2736                96.835236                               3.499896  
               timestamp server_id  cpu_percent  is_anomaly  \
6048 2025-01-01 00:00:00     web_2    32.946949           0   
6049 2025-01-01 00:05:00     web_2    20.723431           0   
6050 2

### Computing Z-score

Z-score is the number of standard deviations a data point is from the mean. 

Computed by the following formula :
$$
z = \frac{x - \mu}{\sigma}
$$

`x` = The raw data point,
$\mu$ = The mean ( average )of the dataset,
$\sigma$ = The standard deviation of the set


In [4]:
def compute_z_score(df, column, windows):
    for window in windows:
        column_name = f'{column}_zscore_{window}'
        roll_std_column_name = f'{column}_roll_standard_deviation_{window}'
        roll_mean_column_name = f'{column}_roll_mean_{window}'
        df[column_name] = np.where(df[roll_std_column_name] == 0, 0, (df[column] - df[roll_mean_column_name])/df[roll_std_column_name])
    return df
    
    

df = compute_z_score(df, 'cpu_percent',[5,10,30])
df = compute_z_score(df, 'memory_percent',[5,10,30])
df = compute_z_score(df, 'disk_io',[5,10,30])

df.head(10)

,timestamp,server_id,server_type,cpu_percent,memory_percent,disk_io,is_anomaly,anomaly_type,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,...,disk_io_roll_standard_deviation_30,cpu_percent_zscore_5,cpu_percent_zscore_10,cpu_percent_zscore_30,memory_percent_zscore_5,memory_percent_zscore_10,memory_percent_zscore_30,disk_io_zscore_5,disk_io_zscore_10,disk_io_zscore_30
0,2025-01-01 00:00:00,web_1,web,22.483571,23.274928,63.451172,0,normal,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-01 00:05:00,web_1,web,19.308678,25.694752,43.981028,0,normal,22.483571,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-01-01 00:10:00,web_1,web,23.238443,36.885905,57.091144,0,normal,20.896125,2.244988,...,13.767470,1.043354,1.043354,1.043354,7.247533,7.247533,7.247533,0.245146,0.245146,0.245146
3,2025-01-01 00:15:00,web_1,web,27.615149,35.100183,51.622232,0,normal,21.676897,2.085378,...,9.928172,2.847566,2.847566,2.847566,0.892634,0.892634,0.892634,-0.324217,-0.324217,-0.324217
4,2025-01-01 00:20:00,web_1,web,18.829233,28.184342,33.647097,0,normal,23.161460,3.422705,...,8.264545,-1.265732,-1.265732,-1.265732,-0.304081,-0.304081,-0.304081,-2.467080,-2.467080,-2.467080
5,2025-01-01 00:25:00,web_1,web,18.829315,34.272010,40.813835,0,normal,22.295015,3.541161,...,11.591881,-0.978690,-0.978690,-0.978690,0.750266,0.750266,0.750266,-0.788888,-0.788888,-0.788888
6,2025-01-01 00:30:00,web_1,web,27.896064,31.706079,48.501220,0,normal,21.564164,3.855648,...,11.019753,1.642240,1.781128,1.781128,-0.066662,0.203109,0.203109,0.334697,0.006062,0.006062
7,2025-01-01 00:35:00,web_1,web,23.837174,37.501373,61.122924,0,normal,23.281641,4.464336,...,10.059644,0.124438,0.314411,0.314411,1.263360,1.319715,1.319715,1.604417,1.260379,1.260379
8,2025-01-01 00:40:00,web_1,web,17.652628,30.372548,54.637231,0,normal,23.401387,4.470913,...,10.336069,-1.285813,-1.390598,-1.390598,-0.838480,-0.226545,-0.226545,0.715064,0.445856,0.445856
9,2025-01-01 00:45:00,web_1,web,22.712800,35.128161,38.324992,0,normal,21.408883,4.341611,...,9.789777,0.300330,0.137064,0.137064,0.755424,0.738209,0.738209,-0.865451,-1.247821,-1.247821


In [5]:
print(
    df[(df['server_id'] == 'web_1') & (df['is_anomaly'] == 1)]
    [['timestamp', 'server_id','cpu_percent', 'is_anomaly', 'cpu_percent_roll_mean_5','cpu_percent_roll_standard_deviation_5','cpu_percent_zscore_5']]
    .head()
)

print(
    df[df['server_id'] == 'web_2']
    [['timestamp', 'server_id', 'cpu_percent', 'is_anomaly', 'cpu_percent_roll_mean_5','cpu_percent_roll_standard_deviation_5','cpu_percent_zscore_5']]
    .head()
)

               timestamp server_id  cpu_percent  is_anomaly  \
2712 2025-01-10 10:00:00     web_1    93.206469           1   
2713 2025-01-10 10:05:00     web_1    95.118247           1   
2714 2025-01-10 10:10:00     web_1    86.219212           1   
2715 2025-01-10 10:15:00     web_1    92.788924           1   
2716 2025-01-10 10:20:00     web_1    90.476344           1   

      cpu_percent_roll_mean_5  cpu_percent_roll_standard_deviation_5  \
2712                53.899682                              10.983994   
2713                62.493170                              20.279148   
2714                72.219648                              22.249243   
2715                76.336664                              22.627333   
2716                86.531306                              12.321851   

      cpu_percent_zscore_5  
2712              3.578551  
2713              1.608799  
2714              0.629215  
2715              0.727097  
2716              0.320166  
              

In [6]:
df[
    (df['server_id'] == 'web_1') &
    (df['timestamp'] >= '2025-01-10 09:50:00') &
    (df['timestamp'] <= '2025-01-10 11:50:00')
][[
    'timestamp',
    'cpu_percent',
    'is_anomaly',
    'cpu_percent_roll_mean_5',
    'cpu_percent_roll_standard_deviation_5',
    'cpu_percent_zscore_5'
]]

,timestamp,cpu_percent,is_anomaly,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,cpu_percent_zscore_5
2710,2025-01-10 09:50:00,41.815713,0,54.851757,7.912885,-1.647445
2711,2025-01-10 09:55:00,65.323677,0,51.124723,8.938530,1.588511
2712,2025-01-10 10:00:00,93.206469,1,53.899682,10.983994,3.578551
2713,2025-01-10 10:05:00,95.118247,1,62.493170,20.279148,1.608799
2714,2025-01-10 10:10:00,86.219212,1,72.219648,22.249243,0.629215
2715,2025-01-10 10:15:00,92.788924,1,76.336664,22.627333,0.727097
2716,2025-01-10 10:20:00,90.476344,1,86.531306,12.321851,0.320166
2717,2025-01-10 10:25:00,97.761290,1,91.561839,3.412497,1.816690
2718,2025-01-10 10:30:00,92.199874,1,92.472804,4.420397,-0.061743
2719,2025-01-10 10:35:00,96.933005,1,91.889129,4.169304,1.209764


### Computing time-based features

In [7]:
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek

df.head(10)

,timestamp,server_id,server_type,cpu_percent,memory_percent,disk_io,is_anomaly,anomaly_type,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,...,cpu_percent_zscore_10,cpu_percent_zscore_30,memory_percent_zscore_5,memory_percent_zscore_10,memory_percent_zscore_30,disk_io_zscore_5,disk_io_zscore_10,disk_io_zscore_30,hour,day_of_week
0,2025-01-01 00:00:00,web_1,web,22.483571,23.274928,63.451172,0,normal,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,2
1,2025-01-01 00:05:00,web_1,web,19.308678,25.694752,43.981028,0,normal,22.483571,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,2
2,2025-01-01 00:10:00,web_1,web,23.238443,36.885905,57.091144,0,normal,20.896125,2.244988,...,1.043354,1.043354,7.247533,7.247533,7.247533,0.245146,0.245146,0.245146,0,2
3,2025-01-01 00:15:00,web_1,web,27.615149,35.100183,51.622232,0,normal,21.676897,2.085378,...,2.847566,2.847566,0.892634,0.892634,0.892634,-0.324217,-0.324217,-0.324217,0,2
4,2025-01-01 00:20:00,web_1,web,18.829233,28.184342,33.647097,0,normal,23.161460,3.422705,...,-1.265732,-1.265732,-0.304081,-0.304081,-0.304081,-2.467080,-2.467080,-2.467080,0,2
5,2025-01-01 00:25:00,web_1,web,18.829315,34.272010,40.813835,0,normal,22.295015,3.541161,...,-0.978690,-0.978690,0.750266,0.750266,0.750266,-0.788888,-0.788888,-0.788888,0,2
6,2025-01-01 00:30:00,web_1,web,27.896064,31.706079,48.501220,0,normal,21.564164,3.855648,...,1.781128,1.781128,-0.066662,0.203109,0.203109,0.334697,0.006062,0.006062,0,2
7,2025-01-01 00:35:00,web_1,web,23.837174,37.501373,61.122924,0,normal,23.281641,4.464336,...,0.314411,0.314411,1.263360,1.319715,1.319715,1.604417,1.260379,1.260379,0,2
8,2025-01-01 00:40:00,web_1,web,17.652628,30.372548,54.637231,0,normal,23.401387,4.470913,...,-1.390598,-1.390598,-0.838480,-0.226545,-0.226545,0.715064,0.445856,0.445856,0,2
9,2025-01-01 00:45:00,web_1,web,22.712800,35.128161,38.324992,0,normal,21.408883,4.341611,...,0.137064,0.137064,0.755424,0.738209,0.738209,-0.865451,-1.247821,-1.247821,0,2


### Rate of change 

Essentially calculating how much did this value change from the previous reading for each `server_id`.

In [8]:
def calculate_rate_of_change(df,column):
    column_name = f'{column}_rate_of_change'
    df[column_name] = df.groupby('server_id')[column].diff()
    return df

df = calculate_rate_of_change(df,'cpu_percent')
df = calculate_rate_of_change(df,'memory_percent')
df = calculate_rate_of_change(df,'disk_io')
df.head(10)

,timestamp,server_id,server_type,cpu_percent,memory_percent,disk_io,is_anomaly,anomaly_type,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,...,memory_percent_zscore_10,memory_percent_zscore_30,disk_io_zscore_5,disk_io_zscore_10,disk_io_zscore_30,hour,day_of_week,cpu_percent_rate_of_change,memory_percent_rate_of_change,disk_io_rate_of_change
0,2025-01-01 00:00:00,web_1,web,22.483571,23.274928,63.451172,0,normal,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0,2,NaN,NaN,NaN
1,2025-01-01 00:05:00,web_1,web,19.308678,25.694752,43.981028,0,normal,22.483571,NaN,...,NaN,NaN,NaN,NaN,NaN,0,2,-3.174892,2.419824,-19.470143
2,2025-01-01 00:10:00,web_1,web,23.238443,36.885905,57.091144,0,normal,20.896125,2.244988,...,7.247533,7.247533,0.245146,0.245146,0.245146,0,2,3.929764,11.191153,13.110116
3,2025-01-01 00:15:00,web_1,web,27.615149,35.100183,51.622232,0,normal,21.676897,2.085378,...,0.892634,0.892634,-0.324217,-0.324217,-0.324217,0,2,4.376707,-1.785722,-5.468912
4,2025-01-01 00:20:00,web_1,web,18.829233,28.184342,33.647097,0,normal,23.161460,3.422705,...,-0.304081,-0.304081,-2.467080,-2.467080,-2.467080,0,2,-8.785916,-6.915841,-17.975135
5,2025-01-01 00:25:00,web_1,web,18.829315,34.272010,40.813835,0,normal,22.295015,3.541161,...,0.750266,0.750266,-0.788888,-0.788888,-0.788888,0,2,0.000082,6.087668,7.166738
6,2025-01-01 00:30:00,web_1,web,27.896064,31.706079,48.501220,0,normal,21.564164,3.855648,...,0.203109,0.203109,0.334697,0.006062,0.006062,0,2,9.066749,-2.565931,7.687384
7,2025-01-01 00:35:00,web_1,web,23.837174,37.501373,61.122924,0,normal,23.281641,4.464336,...,1.319715,1.319715,1.604417,1.260379,1.260379,0,2,-4.058890,5.795294,12.621705
8,2025-01-01 00:40:00,web_1,web,17.652628,30.372548,54.637231,0,normal,23.401387,4.470913,...,-0.226545,-0.226545,0.715064,0.445856,0.445856,0,2,-6.184546,-7.128825,-6.485694
9,2025-01-01 00:45:00,web_1,web,22.712800,35.128161,38.324992,0,normal,21.408883,4.341611,...,0.738209,0.738209,-0.865451,-1.247821,-1.247821,0,2,5.060172,4.755613,-16.312239


### Remove NaN values
Standard deviation is calculated by 

$$s = \sqrt{\frac{\sum (x_i - \bar{x})^2}{n - 1}}$$

and requires atleast 2 values to be computed, we have 15 servers, exactly 15 first rows and 15 NaN's consistently across the `std` and `Z-score`. Below we drop these rows.

We also have NaN values the `cpu_percent_rate_of_change`, `memory_percent_rate_of_change` and `disk_io_rate_of_change`, on the first row for every `server_id`

In [9]:
df.isna().sum()

timestamp                                     0
server_id                                     0
server_type                                   0
cpu_percent                                   0
memory_percent                                0
disk_io                                       0
is_anomaly                                    0
anomaly_type                                  0
cpu_percent_roll_mean_5                      15
cpu_percent_roll_standard_deviation_5        30
cpu_percent_roll_mean_10                     15
cpu_percent_roll_standard_deviation_10       30
cpu_percent_roll_mean_30                     15
cpu_percent_roll_standard_deviation_30       30
memory_percent_roll_mean_5                   15
memory_percent_roll_standard_deviation_5     30
memory_percent_roll_mean_10                  15
memory_percent_roll_standard_deviation_10    30
memory_percent_roll_mean_30                  15
memory_percent_roll_standard_deviation_30    30
disk_io_roll_mean_5                     

In [10]:
print(len(df))
df = df.dropna()
print(len(df))
df.isna().sum()

90720
90690


timestamp                                    0
server_id                                    0
server_type                                  0
cpu_percent                                  0
memory_percent                               0
disk_io                                      0
is_anomaly                                   0
anomaly_type                                 0
cpu_percent_roll_mean_5                      0
cpu_percent_roll_standard_deviation_5        0
cpu_percent_roll_mean_10                     0
cpu_percent_roll_standard_deviation_10       0
cpu_percent_roll_mean_30                     0
cpu_percent_roll_standard_deviation_30       0
memory_percent_roll_mean_5                   0
memory_percent_roll_standard_deviation_5     0
memory_percent_roll_mean_10                  0
memory_percent_roll_standard_deviation_10    0
memory_percent_roll_mean_30                  0
memory_percent_roll_standard_deviation_30    0
disk_io_roll_mean_5                          0
disk_io_roll_

### Adding server type
This would be an encoded categorical feature, this will allow the model to account for type differences.

In [11]:
df = pd.get_dummies(df,columns=['server_type'],dtype=int)
df.head()

,timestamp,server_id,cpu_percent,memory_percent,disk_io,is_anomaly,anomaly_type,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,cpu_percent_roll_mean_10,...,hour,day_of_week,cpu_percent_rate_of_change,memory_percent_rate_of_change,disk_io_rate_of_change,server_type_batch_worker,server_type_cache,server_type_database,server_type_load_balancer,server_type_web
2,2025-01-01 00:10:00,web_1,23.238443,36.885905,57.091144,0,normal,20.896125,2.244988,20.896125,...,0,2,3.929764,11.191153,13.110116,0,0,0,0,1
3,2025-01-01 00:15:00,web_1,27.615149,35.100183,51.622232,0,normal,21.676897,2.085378,21.676897,...,0,2,4.376707,-1.785722,-5.468912,0,0,0,0,1
4,2025-01-01 00:20:00,web_1,18.829233,28.184342,33.647097,0,normal,23.161460,3.422705,23.161460,...,0,2,-8.785916,-6.915841,-17.975135,0,0,0,0,1
5,2025-01-01 00:25:00,web_1,18.829315,34.272010,40.813835,0,normal,22.295015,3.541161,22.295015,...,0,2,0.000082,6.087668,7.166738,0,0,0,0,1
6,2025-01-01 00:30:00,web_1,27.896064,31.706079,48.501220,0,normal,21.564164,3.855648,21.717398,...,0,2,9.066749,-2.565931,7.687384,0,0,0,0,1


### Rolling window limitation: sustained anomalies get absorbed into their own baseline

When comparing `cpu_zscore_5` against `cpu_zscore_30` for `web_1`'s injected 2-hour CPU spike
(2025-01-10, 10:00–12:00), a clear limitation of rolling-window-based Z-scores becomes visible.

With a short window (5), the Z-score is highest at the very start of the anomaly (~1.5), but
drops toward zero — and even goes negative — within about 25 minutes. This happens because the
rolling mean/std are recalculated from the last 5 readings, which very quickly become entirely
made up of anomalous values. The window "absorbs" the anomaly and starts treating it as the new
normal, even though the underlying issue is still ongoing.

A longer window (30) is more resistant to this — the Z-score stays positive and clearly elevated
(roughly 0.3–2.6) for the full 2-hour window, since the anomaly can't fully saturate 30 readings
within that time. However, even this longer window shows a gradual decline in Z-score over the
duration of the anomaly, as more anomalous readings enter the window.

**Takeaway:** short rolling windows are effective at flagging the *onset* of a sustained anomaly,
but rapidly lose sensitivity the longer that anomaly continues, because the baseline they compare
against is itself built from recent — and increasingly contaminated — data. Longer windows delay
this effect but don't eliminate it. Real-world mitigations exist (e.g. freezing the baseline once
an anomaly is flagged, or comparing against a longer, historically separate reference window), but
were considered out of scope for this project; the limitation is instead documented here and
factored into how detection thresholds are chosen.

In [12]:
df[(df['server_id'] == 'web_1') & (df['is_anomaly'] == 1)][['timestamp', 'cpu_percent', 'cpu_percent_roll_mean_30', 'cpu_percent_roll_standard_deviation_30', 'cpu_percent_zscore_30']]

,timestamp,cpu_percent,cpu_percent_roll_mean_30,cpu_percent_roll_standard_deviation_30,cpu_percent_zscore_30
2712,2025-01-10 10:00:00,93.206469,35.202265,18.796425,3.085917
2713,2025-01-10 10:05:00,95.118247,37.697389,21.285964,2.697593
2714,2025-01-10 10:10:00,86.219212,40.079352,23.539806,1.960078
2715,2025-01-10 10:15:00,92.788924,42.230816,24.719715,2.045255
2716,2025-01-10 10:20:00,90.476344,44.604421,26.051370,1.760826
2717,2025-01-10 10:25:00,97.761290,46.875421,26.996571,1.884901
2718,2025-01-10 10:30:00,92.199874,49.723392,27.722607,1.532197
2719,2025-01-10 10:35:00,96.933005,52.003924,28.322720,1.586327
2720,2025-01-10 10:40:00,97.395977,54.466279,28.924532,1.484197
2721,2025-01-10 10:45:00,97.640736,57.215592,28.955523,1.396112


In [13]:
df[(df['server_id'] == 'web_1') & (df['is_anomaly'] == 1)][['timestamp', 'cpu_percent', 'cpu_percent_roll_mean_5', 'cpu_percent_roll_standard_deviation_5', 'cpu_percent_zscore_5']]

,timestamp,cpu_percent,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,cpu_percent_zscore_5
2712,2025-01-10 10:00:00,93.206469,53.899682,10.983994,3.578551
2713,2025-01-10 10:05:00,95.118247,62.493170,20.279148,1.608799
2714,2025-01-10 10:10:00,86.219212,72.219648,22.249243,0.629215
2715,2025-01-10 10:15:00,92.788924,76.336664,22.627333,0.727097
2716,2025-01-10 10:20:00,90.476344,86.531306,12.321851,0.320166
2717,2025-01-10 10:25:00,97.761290,91.561839,3.412497,1.816690
2718,2025-01-10 10:30:00,92.199874,92.472804,4.420397,-0.061743
2719,2025-01-10 10:35:00,96.933005,91.889129,4.169304,1.209764
2720,2025-01-10 10:40:00,97.395977,94.031887,3.157057,1.065578
2721,2025-01-10 10:45:00,97.640736,94.953298,3.368798,0.797744


### Save the new dataset

In [14]:
df.to_csv('../data/features.csv', index=False)